# 후보 생성 파이프라인 — 팀 공동 담당 작업 참고본

- 팀 프로젝트: Watchback, 공동 담당 이슈 #75 (ShinDuck · Justmagic46)
- 원본 구현 작성자: Justmagic46. ShinDuck의 단독 작성 코드로 표기하지 않습니다.
- 출처: https://github.com/dt4-proj3-team3/watchback/pull/140
- 원본 커밋: bd23e02aed658b52ff8d717908712d3d1e001b12 (미병합 PR)
- 이관 변경: 실행 출력·횟수·메타데이터 제거, 팀 Lakehouse 이름을 portfolio_lakehouse 예시로 치환. 처리 로직은 유지했습니다.
- 실행 조건: Microsoft Fabric/PySpark, Silver 입력 테이블과 rules.rule_queries_v1 모듈을 별도로 준비해야 합니다. 해당 룰 모듈과 데이터는 이 파일에 포함되지 않습니다.
- WRITE_OUTPUT_TABLES=False 기본값을 유지합니다. True로 바꾸면 설정된 대상 테이블에 MERGE 쓰기가 실행됩니다.
- 이관 시 JSON 및 Python 코드 셀 문법만 검증했습니다. Spark SQL·전체 Pipeline 실행과 원본의 검증 성과는 재검증하지 않았습니다.



# DS-04 Gold Re-review Candidate Pipeline (v3 - Clean Production Edition)

OneLake 공식 룰 모듈(`Files/rules/rule_queries_v1.py`)을 동적 임포트하여 위험 신호를 평가하고,
신규 취약점 Baseline Fallback 및 30일 스마트 Dedup을 거쳐 `DS.gold_rereview_*` 테이블에 안전하게 적재합니다.
- 1 Candidate = 1 CVE 단위 매칭 (다중 패키지 및 영향 범위는 array/목록 집계)
- Historical Candidate 규격(cand-, trigger_types, signal, value_type) 100% 일치

## 1. Runtime parameters & Rule Import

In [ ]:
import sys
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F

# 1. OneLake 공식 룰 모듈 및 정책 버전 동적 임포트
sys.path.append("/lakehouse/default/Files")
import rules.rule_queries_v1 as rule_module

RULE_QUERIES = rule_module.RULE_QUERIES
POLICY_VERSION = getattr(rule_module, "POLICY_VERSION", "ver.1")  # py 파일에서 자동 동기화
print(f"✅ OneLake 공식 룰 로드 완료 (총 {len(RULE_QUERIES)}개 규칙, 정책 버전: {POLICY_VERSION})")

# 2. 실행 파라미터 및 대상 테이블 정의
SOURCE_VIEW = "silver_ds_vulnerability_signal_snapshot"
WRITE_OUTPUT_TABLES = False  # 테스트 시 False, 실제 적재 시 True
EXIT_NOTEBOOK = False

SCHEMA_VERSION = "gold.rereview.v0"

DS_BATCH_TABLE = "portfolio_lakehouse.DS.gold_rereview_batch"
DS_CANDIDATE_TABLE = "portfolio_lakehouse.DS.gold_rereview_candidate"
DS_EVIDENCE_TABLE = "portfolio_lakehouse.DS.gold_rereview_candidate_evidence"

now = datetime.now(timezone.utc)
RUN_ID = f"manual-{now.strftime('%Y%m%dT%H%M%SZ')}"
BATCH_ID = f"gold-rereview-{now.strftime('%Y%m%dT%H%M%SZ')}"
GENERATED_AT_UTC = now.strftime("%Y-%m-%d %H:%M:%S")

## 2. Load Silver View & Baseline Fallback

In [ ]:
# 1. Silver 계약 뷰 로드
try:
    raw_df = spark.read.table("silver.silver_ds_vulnerability_signal_snapshot")
except Exception:
    raw_df = spark.read.format("delta").load("Tables/silver/silver_ds_vulnerability_signal_snapshot")

# 2. 룰 쿼리용 표준 뷰 생성 (신규 CVE Baseline Fallback 및 컬럼 충돌 방지)
enhanced_df = raw_df \
    .withColumn("epss_probability_lookback", F.coalesce(F.col("epss_probability_lookback"), F.col("epss_current"))) \
    .withColumn("epss_percentile_lookback", F.coalesce(F.col("epss_percentile_lookback"), F.col("epss_percentile_current"))) \
    .withColumn("has_epss_lookback_baseline", F.coalesce(F.col("has_epss_lookback_baseline"), F.lit(False)) | (F.col("epss_lookback_missing_reason") == "NEW_CVE") | F.col("epss_current").isNotNull())

enhanced_df.createOrReplaceTempView(SOURCE_VIEW)
print(f"✅ 룰 평가용 표준 뷰({SOURCE_VIEW}) 준비 완료 (총 {enhanced_df.count():,}건)")

## 3. Score DS-04 Rules & Trigger Mapping

In [ ]:
# 1. 동적 룰 쿼리 결합 실행
spark.sql(f"CREATE OR REPLACE TEMP VIEW ds04_rule_hits_raw AS\n" + "\nUNION ALL\n".join(RULE_QUERIES.values()))

# 2. Rule ID를 공식 표준 trigger_type으로 정확하게 매핑
spark.sql("""
CREATE OR REPLACE TEMP VIEW ds04_rule_hits AS
SELECT
    h.*,
    CASE rule_id
        WHEN 'R-KEV-NEW' THEN 'KEV_NEW'
        WHEN 'R-EPSS-RISE-D30' THEN 'EPSS_RISE'
        WHEN 'R-EPSS-TOP5-ENTRY-D30' THEN 'EPSS_TOP5_ENTRY'
        WHEN 'R-GHSA-SEV-CRITICAL-CVSS9' THEN 'SEV_CRITICAL_CVSS9'
        WHEN 'R-GHSA-SEV-CRITICAL-XOR-CVSS9' THEN 'SEV_CRITICAL_XOR_CVSS9'
        WHEN 'R-GHSA-SEV-CVSS8' THEN 'SEV_CVSS8'
        WHEN 'R-GHSA-PATCH-NEW' THEN 'PATCH_NEW'
        WHEN 'R-GHSA-PATCH-CHANGE' THEN 'PATCH_CHANGE'
    END AS trigger_type
FROM ds04_rule_hits_raw h
""")
print("✅ 룰 평가 및 트리거 매핑 완료 (ds04_rule_hits)")

## 4. Build Candidates & Evidence Views (1 CVE = 1 Candidate, 자립형 CTE)

In [ ]:
# 1. Candidate 뷰 생성 (1 CVE = 1 Candidate, 패키지 및 영향범위 목록화, 자립형 CTE)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW ds04_candidates AS
WITH rule_scores_by_cve AS (
    SELECT
        COALESCE(cve_id, ghsa_id) AS vuln_key,
        MAX(CASE WHEN axis = 'A' THEN score ELSE 0 END) AS risk_change_score,
        MAX(CASE WHEN axis = 'B' THEN score ELSE 0 END) AS current_severity_score,
        MAX(CASE WHEN axis = 'C' THEN score ELSE 0 END) AS actionability_change_score,
        collect_set(trigger_type) AS trigger_types
    FROM ds04_rule_hits
    GROUP BY COALESCE(cve_id, ghsa_id)
)
SELECT
    '{BATCH_ID}' AS batch_id,
    concat('cand-', substr(sha2(concat_ws('|', COALESCE(cve_id, ghsa_id)), 256), 1, 16)) AS candidate_id,
    MAX(signal_snapshot_date) AS signal_snapshot_date,
    MAX(CAST(epss_lookback_snapshot_date AS date)) AS lookback_snapshot_date,
    first(ghsa_id) AS ghsa_id,
    first(cve_id) AS cve_id,
    array_join(collect_set(package_ecosystem), ', ') AS package_ecosystem,
    array_join(collect_set(package_name), ', ') AS package_name,
    array_join(collect_set(affected_range), ', ') AS affected_range,
    CASE
        WHEN array_contains(rs.trigger_types, 'KEV_NEW') THEN 'P0'
        WHEN (COALESCE(rs.risk_change_score, 0) + COALESCE(rs.current_severity_score, 0) + COALESCE(rs.actionability_change_score, 0)) >= 60 THEN 'P1'
        WHEN (COALESCE(rs.risk_change_score, 0) + COALESCE(rs.current_severity_score, 0) + COALESCE(rs.actionability_change_score, 0)) >= 35 THEN 'P2'
        ELSE 'P3'
    END AS priority,
    (COALESCE(rs.risk_change_score, 0) + COALESCE(rs.current_severity_score, 0) + COALESCE(rs.actionability_change_score, 0)) AS score_total,
    COALESCE(rs.risk_change_score, 0) AS risk_change_score,
    COALESCE(rs.current_severity_score, 0) AS current_severity_score,
    COALESCE(rs.actionability_change_score, 0) AS actionability_change_score,
    rs.trigger_types,
    CASE
        WHEN array_contains(rs.trigger_types, 'KEV_NEW') THEN 'KEV_NEW'
        WHEN array_contains(rs.trigger_types, 'EPSS_TOP5_ENTRY') THEN 'EPSS_TOP5_ENTRY'
        WHEN array_contains(rs.trigger_types, 'EPSS_RISE') THEN 'EPSS_RISE'
        WHEN array_contains(rs.trigger_types, 'PATCH_NEW') THEN 'PATCH_NEW'
        WHEN array_contains(rs.trigger_types, 'PATCH_CHANGE') THEN 'PATCH_CHANGE'
    END AS primary_trigger_type,
    CASE
        WHEN array_contains(rs.trigger_types, 'KEV_NEW') THEN 'CISA KEV 신규 등재로 재검토가 필요합니다.'
        WHEN array_contains(rs.trigger_types, 'EPSS_TOP5_ENTRY') THEN 'EPSS 위험 순위가 30일 전 대비 상위 5% 구간에 진입했습니다.'
        WHEN array_contains(rs.trigger_types, 'EPSS_RISE') THEN 'EPSS가 30일 전 대비 급상승했습니다.'
        WHEN array_contains(rs.trigger_types, 'PATCH_NEW') THEN '새 패치 버전이 확인되었습니다.'
        WHEN array_contains(rs.trigger_types, 'PATCH_CHANGE') THEN '패치 버전 정보가 변경되었습니다.'
    END AS reason_summary,
    MAX(COALESCE(CAST(silver_built_at AS timestamp), CAST(signal_snapshot_date AS timestamp))) AS observed_at,
    timestamp('{GENERATED_AT_UTC}') AS selected_at,
    first(epss_probability_lookback) AS epss_probability_lookback,
    first(epss_current) AS epss_current,
    first(epss_percentile_lookback) AS epss_percentile_lookback,
    first(epss_percentile_current) AS epss_percentile_current,
    first(epss_ratio_d30) AS epss_ratio_d30,
    first(epss_delta_d30) AS epss_delta_d30,
    first(kev_listed_previous) AS kev_listed_previous,
    first(kev_listed_current) AS kev_listed_current,
    first(severity_current) AS severity_current,
    first(cvss_score_current) AS cvss_score_current,
    flatten(collect_set(patched_versions_previous)) AS patched_versions_previous,
    flatten(collect_set(patched_versions_current)) AS patched_versions_current,
    first(epss_snapshot_date) AS epss_snapshot_date,
    first(kev_snapshot_at) AS kev_snapshot_at,
    first(ghsa_snapshot_at) AS ghsa_snapshot_at
FROM silver_ds_vulnerability_signal_snapshot s
JOIN rule_scores_by_cve rs
  ON COALESCE(s.cve_id, s.ghsa_id) = rs.vuln_key
WHERE NOT COALESCE(s.withdrawn_current, false)
  AND (COALESCE(rs.risk_change_score, 0) > 0 OR COALESCE(rs.actionability_change_score, 0) > 0)
GROUP BY COALESCE(s.cve_id, s.ghsa_id), rs.trigger_types, rs.risk_change_score, rs.current_severity_score, rs.actionability_change_score
HAVING score_total >= 15
""")

# 2. Evidence 뷰 생성 (공식 매핑표 준수)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW ds04_candidate_evidence AS
WITH distinct_rule_hits AS (
    SELECT DISTINCT
        COALESCE(cve_id, ghsa_id) AS vuln_key,
        rule_id
    FROM ds04_rule_hits
),
mapped_evidence AS (
    SELECT
        c.batch_id,
        c.candidate_id,
        h.rule_id,
        CASE 
            WHEN h.rule_id = 'R-KEV-NEW' THEN 'kev'
            WHEN h.rule_id = 'R-EPSS-RISE-D30' THEN 'epss_score'
            WHEN h.rule_id = 'R-EPSS-TOP5-ENTRY-D30' THEN 'epss_percentile'
            WHEN h.rule_id IN ('R-GHSA-SEV-CRITICAL-CVSS9', 'R-GHSA-SEV-CRITICAL-XOR-CVSS9', 'R-GHSA-SEV-CVSS8') THEN 'severity_cvss'
            WHEN h.rule_id IN ('R-GHSA-PATCH-NEW', 'R-GHSA-PATCH-CHANGE') THEN 'patch'
        END AS signal,
        CASE 
            WHEN h.rule_id = 'R-KEV-NEW' THEN 'false'
            WHEN h.rule_id = 'R-EPSS-RISE-D30' THEN CAST(c.epss_probability_lookback AS string)
            WHEN h.rule_id = 'R-EPSS-TOP5-ENTRY-D30' THEN CAST(c.epss_percentile_lookback AS string)
            WHEN h.rule_id LIKE 'R-GHSA-PATCH%' THEN array_join(c.patched_versions_previous, ', ')
        END AS previous_value,
        CASE 
            WHEN h.rule_id = 'R-KEV-NEW' THEN 'true'
            WHEN h.rule_id = 'R-EPSS-RISE-D30' THEN CAST(c.epss_current AS string)
            WHEN h.rule_id = 'R-EPSS-TOP5-ENTRY-D30' THEN CAST(c.epss_percentile_current AS string)
            WHEN h.rule_id LIKE 'R-GHSA-SEV%' THEN concat(COALESCE(c.severity_current, ''), '|', CAST(c.cvss_score_current AS string))
            WHEN h.rule_id LIKE 'R-GHSA-PATCH%' THEN array_join(c.patched_versions_current, ', ')
        END AS current_value,
        CASE 
            WHEN h.rule_id = 'R-KEV-NEW' THEN 'boolean'
            WHEN h.rule_id = 'R-EPSS-RISE-D30' THEN 'decimal'
            WHEN h.rule_id = 'R-EPSS-TOP5-ENTRY-D30' THEN 'decimal'
            WHEN h.rule_id LIKE 'R-GHSA-PATCH%' THEN 'array'
            ELSE 'string'
        END AS value_type,
        c.observed_at AS evidence_observed_at,
        CASE 
            WHEN h.rule_id = 'R-KEV-NEW' THEN 'CISA KEV'
            WHEN h.rule_id LIKE 'R-EPSS%' THEN 'FIRST EPSS'
            ELSE 'GitHub Advisory'
        END AS source,
        c.ghsa_snapshot_at AS source_snapshot_at,
        c.signal_snapshot_date AS source_snapshot_date,
        CASE 
            WHEN h.rule_id = 'R-KEV-NEW' THEN concat('CISA KEV에 ', date_format(c.signal_snapshot_date, 'yyyy-MM-dd'), ' 신규 등재되었습니다.')
            WHEN h.rule_id = 'R-EPSS-RISE-D30' THEN concat('EPSS가 30일 전 대비 ', CAST(round(c.epss_ratio_d30, 2) AS string), '배 상승했습니다. (', CAST(round(c.epss_probability_lookback, 4) AS string), ' -> ', CAST(round(c.epss_current, 4) AS string), ')')
            WHEN h.rule_id = 'R-EPSS-TOP5-ENTRY-D30' THEN concat('EPSS 위험 순위가 30일 전 대비 상위 5% 구간에 진입했습니다. (', CAST(round(c.epss_percentile_lookback, 4) AS string), ' -> ', CAST(round(c.epss_percentile_current, 4) AS string), ')')
            WHEN h.rule_id = 'R-GHSA-SEV-CRITICAL-CVSS9' THEN 'CRITICAL 심각도 및 CVSS 9.0 이상 고위험 취약점입니다.'
            WHEN h.rule_id = 'R-GHSA-SEV-CRITICAL-XOR-CVSS9' THEN 'CRITICAL 심각도 또는 CVSS 9.0 이상 취약점입니다.'
            WHEN h.rule_id = 'R-GHSA-SEV-CVSS8' THEN 'CVSS 8.0 이상 범위의 고위험 취약점입니다.'
            WHEN h.rule_id = 'R-GHSA-PATCH-NEW' THEN concat('신규 패치 버전이 출시되었습니다: ', array_join(c.patched_versions_current, ', '))
            WHEN h.rule_id = 'R-GHSA-PATCH-CHANGE' THEN concat('패치 버전 정보가 변경되었습니다: ', array_join(c.patched_versions_current, ', '))
        END AS evidence_summary
    FROM ds04_candidates c
    JOIN distinct_rule_hits h
      ON COALESCE(c.cve_id, c.ghsa_id) = h.vuln_key
)
SELECT
    batch_id,
    candidate_id,
    concat('ev_', substr(sha2(concat_ws('|', candidate_id, rule_id, signal, CAST(source_snapshot_date AS string), COALESCE(previous_value, ''), COALESCE(current_value, '')), 256), 1, 16)) AS evidence_id,
    rule_id,
    signal,
    previous_value,
    current_value,
    value_type,
    evidence_observed_at,
    source,
    source_snapshot_at,
    source_snapshot_date,
    evidence_summary
FROM mapped_evidence
""")
print("✅ Candidate 및 Evidence 뷰 준비 완료 (1 CVE = 1 Candidate)")

## 5. Storage with 30-Day Dedup & Integrity Verification

In [ ]:
if WRITE_OUTPUT_TABLES:
    # 1. Candidate 적재 (1 CVE 단위 candidate_id 기준 중복 방지 MERGE)
    spark.sql(f"""
    MERGE INTO {DS_CANDIDATE_TABLE} AS target
    USING ds04_candidates AS source
    ON target.candidate_id = source.candidate_id
    WHEN NOT MATCHED THEN INSERT *
    """)

    # 2. DS 30일 스마트 Dedup (최근 30일 이내 미전달된 신규 신호만 선별)
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW ds04_deduped_today_evidence AS
    WITH recent_delivered AS (
        SELECT DISTINCT e.candidate_id, e.rule_id
        FROM {DS_EVIDENCE_TABLE} e
        JOIN {DS_BATCH_TABLE} b ON e.batch_id = b.batch_id
        WHERE b.status = 'completed'
          AND b.batch_id <> '{BATCH_ID}'
          AND e.evidence_observed_at >= date_sub(current_date(), 30)
    )
    SELECT src.*
    FROM ds04_candidate_evidence src
    LEFT JOIN recent_delivered r
      ON src.candidate_id = r.candidate_id AND src.rule_id = r.rule_id
    WHERE r.candidate_id IS NULL
    """)

    # 3. Evidence 적재 (선별된 증분 근거만 MERGE 적재)
    spark.sql(f"""
    MERGE INTO {DS_EVIDENCE_TABLE} AS target
    USING ds04_deduped_today_evidence AS source
    ON target.batch_id = source.batch_id
   AND target.candidate_id = source.candidate_id
   AND target.rule_id = source.rule_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)

# 4. 데이터 무결성 검증 (Assert)
cand_df = spark.table(DS_CANDIDATE_TABLE if WRITE_OUTPUT_TABLES else "ds04_candidates")
ev_df = spark.table(DS_EVIDENCE_TABLE if WRITE_OUTPUT_TABLES else "ds04_candidate_evidence").filter(F.col("batch_id") == BATCH_ID if WRITE_OUTPUT_TABLES else "1=1")

assert cand_df.where("candidate_id IS NULL OR priority IS NULL OR primary_trigger_type IS NULL").count() == 0, "Candidate key fields must not be null"
assert ev_df.join(cand_df.select("candidate_id"), ["candidate_id"], "left_anti").count() == 0, "Evidence must not reference missing candidates"

current_batch_candidate_count = ev_df.select("candidate_id").distinct().count()
print(f"🎯 이번 배치({BATCH_ID})에서 Web팀에 새로 전달될 고유 Candidate 수: {current_batch_candidate_count:,}개")

# 5. 최종 Batch ('completed') 기록
if WRITE_OUTPUT_TABLES:
    spark.sql(f"""
    MERGE INTO {DS_BATCH_TABLE} AS target
    USING (
        SELECT
            '{BATCH_ID}' AS batch_id,
            '{RUN_ID}' AS ds_run_id,
            'completed' AS status,
            max(observed_at) AS as_of,
            timestamp('{GENERATED_AT_UTC}') AS generated_at,
            '{POLICY_VERSION}' AS policy_version,
            '{SCHEMA_VERSION}' AS schema_version,
            CAST({current_batch_candidate_count} AS int) AS candidate_count
        FROM ds04_candidates
    ) AS source
    ON target.batch_id = source.batch_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    display(spark.table(DS_BATCH_TABLE).filter(F.col("batch_id") == BATCH_ID))
else:
    print("ℹ️ [WRITE_OUTPUT_TABLES=False] 테스트 모드: 실제 저장이 생략되었습니다.")

## 5-1. Candidate, Evidence, Batch 3개 테이블 및 우선순위 집계 확인

In [ ]:
# 1. 우선순위(Priority)별 집계 표
print("📊 [우선순위별 집계]")
display(spark.sql("""
SELECT 
    priority,
    COUNT(*) AS candidate_count,
    ROUND(AVG(score_total), 1) AS avg_score
FROM ds04_candidates
GROUP BY priority
ORDER BY priority
"""))

# 2. Batch 테이블 미리보기 (즉석 뷰 생성하여 무조건 100% 출력 보장!)
print("\n1️⃣ [Batch 테이블]")
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW ds04_batch_preview AS
SELECT
    '{BATCH_ID}' AS batch_id,
    '{RUN_ID}' AS ds_run_id,
    'completed' AS status,
    MAX(observed_at) AS as_of,
    timestamp('{GENERATED_AT_UTC}') AS generated_at,
    '{POLICY_VERSION}' AS policy_version,
    '{SCHEMA_VERSION}' AS schema_version,
    COUNT(DISTINCT candidate_id) AS candidate_count
FROM ds04_candidates
""")
display(spark.table("ds04_batch_preview"))

# 3. Candidate 테이블 확인
print("\n2️⃣ [Candidate 테이블]")
display(spark.table("ds04_candidates"))

# 4. Evidence 테이블 확인
print("\n3️⃣ [Evidence 테이블]")
display(spark.table("ds04_candidate_evidence"))

## 6. Execution Summary

In [ ]:
run_result = {
    "status": "ok",
    "run_id": RUN_ID,
    "batch_id": BATCH_ID,
    "candidate_count": current_batch_candidate_count,
    "write_output_tables": WRITE_OUTPUT_TABLES
}
print(f"🎯 DS-04 Pipeline finished: {run_result}")

if EXIT_NOTEBOOK:
    import notebookutils
    notebookutils.notebook.exit(json.dumps(run_result))